In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('/Users/cono/code/learning_ml/src/ml_learning/makemore_conor/names.txt', 'r').read().splitlines()

In [3]:
len(words)

32033

In [4]:
# We had S/E special tokens, but now just "."
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# ------------ now made respectable :) ---------------

In [6]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words, block_size):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1], block_size)
Xdev, Ydev = build_dataset(words[n1:n2], block_size)
Xte, Yte = build_dataset(words[n2:], block_size)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [7]:
# What if things are truly random???
# Note equality! If we have a uniform model, probability of anything right is 1/27, so you take logs and nll = log(27)
import numpy as np
logits_equal = torch.ones(Ytr.shape[0], 27)
loss = F.cross_entropy(logits_equal, Ytr)
print(loss, np.log(27))

tensor(3.2958) 3.295836866004329


In [27]:
# Adding skip connections
def compute_loss(params, Xtr, Ytr):
    mb_size = params["mb_size"]
    n_embedding = params["n_embedding"]
    #n_context = params["n_context"]
    n_context = 3
    n_hidden = params["n_hidden"]
    learning_rate = params["learning_rate"]
    weight_decay_lambda = params["reg"]
    steps = params["steps"]
    shrinker = 1/mb_size

    fan_in = [n_context * n_embedding, n_hidden]
    fan_out = [n_hidden, 27]
    shrinkers_glorot = [6 / np.sqrt(fan_in[0] + fan_out[0]), 6 / np.sqrt(fan_in[1] + fan_out[1])]
    shrinkers_he = [np.sqrt(2 / fan_in[0]), np.sqrt(2 / fan_in[1])]
    print(shrinkers_he)
    
    # Now define params
    g = torch.Generator().manual_seed(2147483647) # for reproducibility
    C = torch.randn((27, n_embedding), generator=g)  * shrinkers_he[0]
    W1 = torch.randn((n_context * n_embedding, n_hidden), generator=g) * shrinkers_he[0]
    b1 = torch.randn(n_hidden, generator=g) * shrinkers_he[0]
    W2 = torch.randn((n_hidden, 27), generator=g) * shrinkers_he[1]
    b2 = torch.randn(27, generator=g) * shrinkers_he[1]
    Wskip = torch.randn((n_context * n_embedding, 27), generator=g) * shrinkers_he[0]
    parameters = [C, W1, b1, W2, b2, Wskip]
    
    for p in parameters:
        p.requires_grad = True
    
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2 + emb.view(-1, n_embedding * n_context) @ Wskip
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate
        for p in parameters:
            p.data += -lr * p.grad

    small_steps = int(steps / 5)
    for i in range(small_steps):
        ix = torch.randint(0, Xtr.shape[0], (mb_size,))
        # forward pass
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1)
        logits = h @ W2 + b2 + emb.view(-1, n_embedding * n_context) @ Wskip
        loss = F.cross_entropy(logits,Ytr[ix]) + weight_decay_lambda * (W1**2).sum() + weight_decay_lambda * (W2**2).sum()
        print(loss.item())
        # Backward pass - should be none!
        for p in parameters:
            p.grad = None
        
        loss.backward()
        lr = learning_rate / 10
        for p in parameters:
            p.data += -lr * p.grad

    emb = C[Xtr]
    h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) 
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr)

    tuned_params = [C, W1, b1, W2, b2, Wskip]
    return loss, tuned_params

In [28]:
params = {'learning_rate': 0.9954213334422817,
 'n_embedding': 8,
 'mb_size': 1024,
 'n_hidden': 454,
 'reg': 8.13775452119088e-05,
 'steps': 14386}

In [29]:
_, tuned_params = compute_loss(params, Xtr, Ytr)

[np.float64(0.28867513459481287), np.float64(0.0663723311599972)]
3.5239641666412354
3.4710869789123535
4.765206813812256
3.911191940307617
4.919017314910889
4.670767784118652
5.754182815551758
5.277332782745361
5.423601150512695
7.794350624084473
9.318053245544434
5.514338970184326
10.890090942382812
7.269193172454834
4.6395487785339355
3.9507105350494385
6.817602634429932
5.362793922424316
4.253064155578613
5.684527397155762
6.690552234649658
4.503046035766602
3.2710840702056885
3.0220882892608643
3.0644640922546387
3.9421019554138184
3.7680304050445557
3.527599334716797
2.8758881092071533
2.716675043106079
2.715867280960083
2.7670998573303223
2.7635254859924316
2.7243154048919678
2.6248369216918945
2.61033034324646
2.662834644317627
2.7173445224761963
2.668292999267578
2.5747039318084717
2.490792751312256
2.6094908714294434
2.566727638244629
2.5315611362457275
2.5027689933776855
2.611482620239258
2.5704097747802734
2.5364296436309814
2.5112533569335938
2.4900643825531006
2.481296062

In [30]:
C, W1, b1, W2, b2, Wskip = tuned_params

In [31]:
n_embedding = 8
n_context = 3

In [32]:
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, n_embedding * n_context) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 + emb.view(-1, n_embedding * n_context) @ Wskip # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.0842, grad_fn=<NllLossBackward0>)

In [33]:
# Can tune hyperparameters (context, embedding)...
# Use better optimisation?
# He uses 100,000 0.1 and 0.01 - gets 2.16 on dev, 2.12 on train
# Hyperparameters to tune - (C-dimension, context size, number of neurons, proper learning rate?, batch size)
# Read paper and do things

In [35]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2 + emb.view(-1, n_embedding * n_context) @ Wskip
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carmah.
amelle.
khi.
mili.
taty.
skarshie.
rahnen.
delynn.
jareei.
nellara.
chaiivon.
leigh.
ham.
joce.
quint.
shon.
walianni.
waterri.
jarynix.
kael.
